In [ ]:
#@title [준비] 실행 버튼(>)만 눌러주세요
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize

def _gen(seed, x_cfg, y_funcs, n=1000):
    rng = np.random.default_rng(seed)
    xs = {nm: rng.uniform(lo, hi, n) for nm, (lo, hi) in x_cfg.items()}
    ys = {nm: fn(xs, rng) for nm, fn in y_funcs.items()}
    return pd.DataFrame({**xs, **ys})

df = _gen(42,
    {'온도_C': (50,200), '압력_kPa': (100,500), '유량_Lmin': (1,10), '교반_RPM': (100,800), '시간_min': (10,120)},
    {
        '순도_pct': lambda x, r: np.clip(80+0.05*x['온도_C']+0.01*x['압력_kPa']-0.3*x['유량_Lmin']+0.005*x['교반_RPM']+0.02*x['시간_min']-0.0001*x['온도_C']*x['압력_kPa']+0.0002*x['온도_C']*x['교반_RPM']+r.normal(0,1.5,1000), 85, 99.9),
        '입자크기_um': lambda x, r: np.clip(25-0.05*x['온도_C']+0.01*x['압력_kPa']+0.8*x['유량_Lmin']-0.01*x['교반_RPM']-0.03*x['시간_min']+0.0005*x['유량_Lmin']*x['교반_RPM']+r.normal(0,1.0,1000), 1, 50),
        '점도_cP': lambda x, r: np.clip(50+0.3*x['온도_C']+0.1*x['압력_kPa']-2.0*x['유량_Lmin']+0.05*x['교반_RPM']+0.5*x['시간_min']-0.001*x['온도_C']*x['유량_Lmin']+r.normal(0,5,1000), 30, 400),
    }
)

input_cols = ['온도_C', '압력_kPa', '유량_Lmin', '교반_RPM', '시간_min']
output_cols = ['순도_pct', '입자크기_um', '점도_cP']
specs = {'순도_pct': (95, 99.9), '입자크기_um': (5, 20), '점도_cP': (80, 250)}

X_tr, X_te, y_tr, y_te = train_test_split(df[input_cols], df[output_cols], test_size=0.2, random_state=42)
factory_model = RandomForestRegressor(n_estimators=100, random_state=42)
factory_model.fit(X_tr, y_tr)

def find_optimal(targets, n_restarts=30):
    bounds = [(50,200),(100,500),(1,10),(100,800),(10,120)]
    t_arr = np.array([targets[c] for c in output_cols])
    scales = np.array([specs[c][1]-specs[c][0] for c in output_cols])
    def obj(x):
        return np.sqrt(np.sum(((factory_model.predict([x])[0]-t_arr)/scales)**2))
    best = None
    for _ in range(n_restarts):
        x0 = [np.random.uniform(lo,hi) for lo,hi in bounds]
        res = minimize(obj, x0, bounds=bounds, method='L-BFGS-B')
        if best is None or res.fun < best.fun: best = res
    return dict(zip(input_cols, best.x)), dict(zip(output_cols, factory_model.predict([best.x])[0]))

# R2 성능
y_pred = factory_model.predict(X_te)
r2 = {col: r2_score(y_te.iloc[:,i], y_pred[:,i]) for i, col in enumerate(output_cols)}

print('준비 완료!')
print(f'df: 공장 데이터 {len(df)} 로트')
print(f'factory_model: 품질 예측 모델 (Random Forest)')
print(f'find_optimal: 최적 조건 탐색 함수')
print()
for col, score in r2.items():
    print(f'  {col:14s} R2={score:.3f}')
print()
print('# Colab AI에게 시켜보세요:')
print('# "df 처음 10줄 보여줘"')
print('# "온도와 순도의 산점도를 그려줘"')
print('# "factory_model로 온도 150, 압력 300, 유량 5, 교반 400, 시간 60일 때 예측해줘"')
print('# "find_optimal로 순도 97%, 입자크기 10um, 점도 150cP 최적 조건 찾아줘"')